In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

# Find project root
current_dir = Path.cwd()

if (current_dir / ".env").exists():
    PROJECT_DIR = current_dir
else:
    PROJECT_DIR = current_dir.parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
WAREHOUSE_DIR = PROJECT_DIR / "data" / "warehouse"

WAREHOUSE_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Warehouse folder:", WAREHOUSE_DIR)

Project: d:\Atomcamp AI Bootcamp\6. ETL Module\Online-Shop-ETL-Project
Warehouse folder: d:\Atomcamp AI Bootcamp\6. ETL Module\Online-Shop-ETL-Project\data\warehouse


In [2]:
# Load transformed datasets

customers_df = pd.read_csv(
    PROCESSED_DIR / "customers_clean.csv"
)

orders_df = pd.read_csv(
    PROCESSED_DIR / "orders_clean.csv"
)

mongo_df = pd.read_csv(
    PROCESSED_DIR / "mongo_orders_flat.csv"
)

# Convert dates
customers_df["signup_date"] = pd.to_datetime(
    customers_df["signup_date"],
    errors="coerce"
)

orders_df["order_date"] = pd.to_datetime(
    orders_df["order_date"],
    errors="coerce"
)

mongo_df["order_date"] = pd.to_datetime(
    mongo_df["order_date"],
    errors="coerce"
)

print("Customers:", customers_df.shape)
print("Orders:", orders_df.shape)
print("MongoDB app records:", mongo_df.shape)

Customers: (2000, 5)
Orders: (200000, 20)
MongoDB app records: (5000, 16)


In [3]:
# Create the date dimension
date_range = pd.date_range(
    start="2025-01-01",
    end="2026-12-31",
    freq="D"
)

dim_date = pd.DataFrame({
    "date": date_range
})

dim_date["date_key"] = (
    dim_date["date"].dt.strftime("%Y%m%d").astype(int)
)

dim_date["day"] = dim_date["date"].dt.day
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.month_name()
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["year"] = dim_date["date"].dt.year
dim_date["day_name"] = dim_date["date"].dt.day_name()
dim_date["week_of_year"] = (
    dim_date["date"].dt.isocalendar().week.astype(int)
)

dim_date["is_weekend"] = (
    dim_date["date"].dt.dayofweek >= 5
)

print("Date dimension created:", len(dim_date))
display(dim_date.head())

Date dimension created: 730


C:\Users\Super Computer Bannu\AppData\Local\Temp\ipykernel_20124\1380817608.py:27: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dim_date["date"].dt.dayofweek >= 5


,date,date_key,day,month,month_name,quarter,year,day_name,week_of_year,is_weekend
0,2025-01-01,20250101,1,1,January,1,2025,Wednesday,1,False
1,2025-01-02,20250102,2,1,January,1,2025,Thursday,1,False
2,2025-01-03,20250103,3,1,January,1,2025,Friday,1,False
3,2025-01-04,20250104,4,1,January,1,2025,Saturday,1,True
4,2025-01-05,20250105,5,1,January,1,2025,Sunday,1,True


SCD Type 2 means that instead of overwriting a customer's previous information, we preserve historical versions.

Customer 15 --> Karachi --> Old info
Customer 15 --> Lahore ---> Current info


We used:
- PostgreSQL customer records for the current city.
- MongoDB app records for historical city information.
- Order dates to estimate when the city changed.


In [4]:
# Create the customer dimension using SCD Type 2


# Make sure customer IDs have consistent types
customers_df["customer_id"] = pd.to_numeric(
    customers_df["customer_id"],
    errors="coerce"
).astype("Int64")

mongo_df["customer_id"] = pd.to_numeric(
    mongo_df["customer_id"],
    errors="coerce"
).astype("Int64")

# Keep only app records that contain a city
app_customer_history = mongo_df[
    ["customer_id", "customer_city", "order_date"]
].dropna(
    subset=["customer_id", "customer_city", "order_date"]
).copy()

# Current customer information from PostgreSQL
current_customers = customers_df.copy()

customer_versions = []

for customer in current_customers.itertuples(index=False):

    customer_id = int(customer.customer_id)
    current_city = customer.city

    history = app_customer_history[
        app_customer_history["customer_id"] == customer_id
    ].copy()

    # Find app observations with a different city
    old_city_records = history[
        history["customer_city"] != current_city
    ]

    if not old_city_records.empty:

        # Identify the historical city
        old_city = (
            old_city_records
            .sort_values("order_date")
            .iloc[0]["customer_city"]
        )

        # Find the last observed order in the old city
        last_old_date = old_city_records[
            old_city_records["customer_city"] == old_city
        ]["order_date"].max()

        # Approximate transition date
        transition_date = last_old_date + pd.Timedelta(days=1)

        # Old customer version
        customer_versions.append({
            "customer_id": customer_id,
            "name": customer.name,
            "city": old_city,
            "segment": customer.segment,
            "signup_date": customer.signup_date,
            "effective_from": customer.signup_date,
            "effective_to": last_old_date,
            "is_current": False
        })

        # Current customer version
        customer_versions.append({
            "customer_id": customer_id,
            "name": customer.name,
            "city": current_city,
            "segment": customer.segment,
            "signup_date": customer.signup_date,
            "effective_from": transition_date,
            "effective_to": pd.Timestamp("2026-12-31"),
            "is_current": True
        })

    else:

        # Customer has only one observed city
        customer_versions.append({
            "customer_id": customer_id,
            "name": customer.name,
            "city": current_city,
            "segment": customer.segment,
            "signup_date": customer.signup_date,
            "effective_from": customer.signup_date,
            "effective_to": pd.Timestamp("2026-12-31"),
            "is_current": True
        })

dim_customer = pd.DataFrame(customer_versions)

# Assign a unique surrogate key to every version
dim_customer.insert(
    0,
    "customer_key",
    range(1, len(dim_customer) + 1)
)

print("Customer dimension records:", len(dim_customer))
print(
    "Customers with historical versions:",
    dim_customer.groupby("customer_id").size().gt(1).sum()
)

display(dim_customer.head(10))

Customer dimension records: 2057
Customers with historical versions: 57


,customer_key,customer_id,name,city,segment,signup_date,effective_from,effective_to,is_current
0,1,1,Usman Sheikh,Karachi,Consumer,2022-11-28,2022-11-28 00:00:00,2026-12-31 00:00:00,True
1,2,2,Iqra Khan,Faisalabad,Consumer,2022-11-11,2022-11-11 00:00:00,2026-12-31 00:00:00,True
2,3,3,Maryam Raza,Faisalabad,Consumer,2023-10-20,2023-10-20 00:00:00,2026-12-31 00:00:00,True
3,4,4,Ali Sheikh,Lahore,Consumer,2022-08-09,2022-08-09 00:00:00,2026-12-31 00:00:00,True
4,5,5,Hamza Qureshi,Quetta,Consumer,2022-11-10,2022-11-10 00:00:00,2026-04-14 15:04:31,False
5,6,5,Hamza Qureshi,Karachi,Consumer,2022-11-10,2026-04-15 15:04:31,2026-12-31 00:00:00,True
6,7,6,Noor Malik,Islamabad,Consumer,2023-06-25,2023-06-25 00:00:00,2026-12-31 00:00:00,True
7,8,7,Noor Qureshi,Multan,Small Business,2023-02-28,2023-02-28 00:00:00,2026-12-31 00:00:00,True
8,9,8,Usman Khan,Peshawar,Consumer,2022-08-18,2022-08-18 00:00:00,2026-12-31 00:00:00,True
9,10,9,Amna Abbasi,Islamabad,Consumer,2023-12-05,2023-12-05 00:00:00,2026-12-31 00:00:00,True


In [5]:
# Create the product dimension

product_categories = sorted(
    orders_df["category"].dropna().unique()
)

dim_product = pd.DataFrame({
    "product_key": range(1, len(product_categories) + 1),
    "category": product_categories
})

print("Product categories:", len(dim_product))
display(dim_product)

Product categories: 6


,product_key,category
0,1,Beauty
1,2,Books
2,3,Electronics
3,4,Fashion
4,5,Grocery
5,6,Home


In [6]:
# Create the payment dimension

payment_methods = sorted(
    orders_df["payment_method"].dropna().unique()
)

dim_payment = pd.DataFrame({
    "payment_key": range(1, len(payment_methods) + 1),
    "payment_method": payment_methods
})

display(dim_payment)

,payment_key,payment_method
0,1,bank_transfer
1,2,card
2,3,cash_on_delivery
3,4,wallet


In [7]:
# Create the order status dimension

statuses = sorted(
    orders_df["status"].dropna().unique()
)

dim_status = pd.DataFrame({
    "status_key": range(1, len(statuses) + 1),
    "status": statuses
})

display(dim_status)

,status_key,status
0,1,cancelled
1,2,delivered
2,3,pending
3,4,returned
4,5,shipped


In [8]:
# Prepare the fact table

# Date lookup
date_lookup = dim_date.set_index("date")["date_key"]

# Product lookup
product_lookup = dim_product.set_index(
    "category"
)["product_key"]

# Payment lookup
payment_lookup = dim_payment.set_index(
    "payment_method"
)["payment_key"]

# Status lookup
status_lookup = dim_status.set_index(
    "status"
)["status_key"]

In [9]:
# Match each order to its customer version

orders_for_fact = orders_df.copy()

orders_for_fact["order_date"] = pd.to_datetime(
    orders_for_fact["order_date"],
    errors="coerce"
)

# Join customer versions using customer ID
fact_customer = orders_for_fact.merge(
    dim_customer,
    on="customer_id",
    how="left",
    suffixes=("", "_dimension")
)

# Keep the version valid on the order date
fact_customer = fact_customer[
    (
        fact_customer["order_date"]
        >= fact_customer["effective_from"]
    )
    &
    (
        fact_customer["order_date"]
        <= fact_customer["effective_to"]
    )
].copy()

# Check that each order has one matching customer version
print("Orders after customer version matching:", len(fact_customer))

print(
    "Duplicate order IDs after matching:",
    fact_customer["order_id"].duplicated().sum()
)

Orders after customer version matching: 199989
Duplicate order IDs after matching: 0


In [ ]:
# Build the fact table

fact_orders = pd.DataFrame({
    "order_id": fact_customer["order_id"],
    "date_key": fact_customer["order_date"].map(date_lookup),
    "customer_key": fact_customer["customer_key"],
    "product_key": fact_customer["category"].map(product_lookup),
    "payment_key": fact_customer["payment_method"].map(payment_lookup),
    "status_key": fact_customer["status"].map(status_lookup),
    "quantity": fact_customer["quantity"],
    "amount": fact_customer["amount"],
    "source": fact_customer["source"]
})

# Reset index for a clean output
fact_orders = fact_orders.reset_index(drop=True)

print("Fact table created:", fact_orders.shape)
display(fact_orders.head())

Fact table created: (199989, 9)


,order_id,date_key,customer_key,product_key,payment_key,status_key,quantity,amount,source
0,100001,20260901,268,3,1,2,1,77635.24,web
1,100002,20250228,102,5,2,2,1,5562.63,web
2,100003,20260708,802,2,2,5,1,2055.15,web
3,100004,20250222,1549,1,4,2,1,7414.43,web
4,100005,20260112,165,2,4,2,4,6784.53,web


In [11]:
# Validate the warehouse tables

warehouse_checks = {
    "Date dimension contains 730 dates":
        len(dim_date) == 730,

    "Customer dimension has unique surrogate keys":
        dim_customer["customer_key"].is_unique,

    "Product dimension has unique keys":
        dim_product["product_key"].is_unique,

    "Payment dimension has unique keys":
        dim_payment["payment_key"].is_unique,

    "Status dimension has unique keys":
        dim_status["status_key"].is_unique,

    "Fact table has unique order IDs":
        fact_orders["order_id"].is_unique,

    "Fact table has no missing foreign keys":
        fact_orders[
            [
                "date_key",
                "customer_key",
                "product_key",
                "payment_key",
                "status_key"
            ]
        ].notna().all().all(),

    "Fact table has positive amounts":
        (fact_orders["amount"] > 0).all()
}

for check, result in warehouse_checks.items():
    print(
        f"{'PASS' if result else 'FAIL'}: {check}"
    )

print(
    f"\nPassed {sum(warehouse_checks.values())}"
    f" out of {len(warehouse_checks)} checks."
)

PASS: Date dimension contains 730 dates
PASS: Customer dimension has unique surrogate keys
PASS: Product dimension has unique keys
PASS: Payment dimension has unique keys
PASS: Status dimension has unique keys
PASS: Fact table has unique order IDs
PASS: Fact table has no missing foreign keys
PASS: Fact table has positive amounts

Passed 8 out of 8 checks.


In [12]:
# Save the warehouse tables

warehouse_tables = {
    "dim_date": dim_date,
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_payment": dim_payment,
    "dim_status": dim_status,
    "fact_orders": fact_orders
}

for table_name, dataframe in warehouse_tables.items():

    dataframe.to_csv(
        WAREHOUSE_DIR / f"{table_name}.csv",
        index=False
    )

    print(
        f"Saved {table_name}.csv: "
        f"{len(dataframe):,} rows"
    )

print("\nWarehouse tables saved successfully!")

Saved dim_date.csv: 730 rows
Saved dim_customer.csv: 2,057 rows
Saved dim_product.csv: 6 rows
Saved dim_payment.csv: 4 rows
Saved dim_status.csv: 5 rows
Saved fact_orders.csv: 199,989 rows

Warehouse tables saved successfully!


In [13]:
print("========== DATA WAREHOUSE SUMMARY ==========")

for table_name, dataframe in warehouse_tables.items():
    print(f"{table_name}: {len(dataframe):,} rows")

print("\nStar schema creation completed!")

========== DATA WAREHOUSE SUMMARY ==========
dim_date: 730 rows
dim_customer: 2,057 rows
dim_product: 6 rows
dim_payment: 4 rows
dim_status: 5 rows
fact_orders: 199,989 rows

Star schema creation completed!
